In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

In [6]:
#middleware
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [7]:
#run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [11]:
questions=[
    "what is 2+20",
    "what is 2/20",
    "what is 2*20",
    "what is 2-200",
    "what is 200**2",
    "what is 200+2",
]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages: {response}")
    print(f"messages: {len(response['messages'])}")
    

messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n<think>\nOkay, the user is asking for the result of 2 minus 200. Let me think through this step by step.\n\nFirst, subtraction of a larger number from a smaller one. So 2 - 200. Since 200 is bigger than 2, the result will be negative. How much? Well, 2 minus 200. Let me visualize a number line. Starting at 2, moving 200 units to the left. That would take me past 0 into the negative numbers. The total distance from 2 to 200 is 198, so 2 - 200 is -198. Let me check another way: 200 - 2 is 198, so the answer should be -198. \n\nAnother way to think about it: if I have $2 and I spend $200, I'd be in debt by $198. That makes sense. Alternatively, using algebra: 2 - 200 = -(200 - 2) = -198. Yeah, that works. \n\nWait, let me confirm with basic arithmetic. 2 minus 200. If I subtract 2 from both numbers, 2 - 2 = 0 and 200 - 2 = 198, so the difference is -198. Yep, that's correct. \n\nI should make s

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kn4j9a48e6pvbwzrcq9a4ga1` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3894, Requested 2453. Please try again in 3.47s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [13]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [14]:
def read_email_tool(email_id:str)->str:
    """Mock function to read and email by its id"""
    return f"Email content is for id:{email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send and email"""
    return f"Email send to {recipient} with subject :{subject}"


In [16]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decision":["approved","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [22]:
config={"configurable":{"thread_id":"test-approve"}}
result=agent.invoke(
    {
        "messages":[HumanMessage(content="send email to m@m.com with subject hi how are yoy")],
        
    },config=config
)

In [19]:
result

{'messages': [HumanMessage(content='send email to m@m.com with subject hi how are yoy', additional_kwargs={}, response_metadata={}, id='756258e0-c55f-49c8-a512-5a8bf0bb1156'),
  AIMessage(content="Please provide the body/content of the email you'd like me to send.", additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to m@m.com with the subject "hi how are yoy". Let me check the available tools. There\'s the send_email_tool which requires recipient, subject, and body. The user provided the recipient and subject, but the body isn\'t mentioned. I need to ask for the email body. Since the subject is "hi how are yoy", maybe the body is a greeting, but I can\'t assume. I should prompt the user to provide the body content to complete the request.\n'}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 246, 'total_tokens': 374, 'completion_time': 0.230501306, 'completion_tokens_details': {'reasoning_tokens': 108}, 'prompt_time': 0.01057175

In [23]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("approval pending")
    result=agent.invoke(
        Command(
            resume={
                "decision":[
                    {"type":"approve"}
                ]
            }
        ),config=config
    )
    print(f"result: {result['message'][-1].content}")